[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HumbertoDiego/AjustamentoBasicoIME/blob/main/06_cond_sistemas.ipynb)

# Ajustamento Básico - Sistemas mal condicionados

**Maj Diego - 2° Semestre / 2026**

**Objetivos:**

1. Analisar sistemas com comportamento divergente
2. Decomposição SVD
3. Matrizes pseudoinversas

**Referência:** Gilbert Strang, Kai Borre (1997). *Linear Algebra, Geodesy, and GPS*. Wellesley-Cambridge Press. $\rightarrow$ **Cap 11**


## O Problema

Existem situações em que um sistema linear fica mal condicionado, ou seja, pequenas variações nas observações podem gerar grandes variações nas soluções:

|$\begin{matrix} \text{Sistema} & AX&=&b\end{matrix} $|$\begin{matrix}\text{Solução} & \ & \ & \ & \end{matrix}$ | $\text{Comportamento}$|
|-|-|-|
|$ \begin{bmatrix} 1 & 1 \\ 1 &-2\end{bmatrix} \begin{bmatrix} x_1 \\ x_2\end{bmatrix} =  \begin{bmatrix} 10 \\ -5\end{bmatrix}$ | $ \begin{bmatrix}x_1 \\ x_2\end{bmatrix}= \begin{bmatrix}5 \\ 5\end{bmatrix}$ |Convergente|
|$ \begin{bmatrix} 1 & 1 \\ 1 &-2\end{bmatrix} \begin{bmatrix} x_1 \\ x_2\end{bmatrix} =  \begin{bmatrix} 10.01 \\ -5\end{bmatrix}$ | $ \begin{bmatrix}x_1 \\ x_2\end{bmatrix}= \begin{bmatrix}5.007 \\ 5.003 \end{bmatrix}$ |Convergente|
|$ \begin{bmatrix} 1 & 1 \\ 1.001 & 1\end{bmatrix} \begin{bmatrix} x_1 \\ x_2\end{bmatrix} =  \begin{bmatrix} 10 \\ 10.005\end{bmatrix}$ | $ \begin{bmatrix}x_1 \\ x_2\end{bmatrix}= \begin{bmatrix}5 \\ 5 \end{bmatrix}$ |Divergente|
|$ \begin{bmatrix} 1 & 1 \\ 1.001 & 1\end{bmatrix} \begin{bmatrix}x_1 \\ x_2\end{bmatrix} =  \begin{bmatrix} 10 \\ 10.1\end{bmatrix}$ | $ \begin{bmatrix}x_1 \\ x_2\end{bmatrix}= \begin{bmatrix}100 \\ -90 \end{bmatrix}$ |Divergente|

## 1. Analisar sistemas com comportamento divergente

### **1.1 As causas do comportamento divergente de um sistema linear**

O **condicionamento** de um sistema linear ($AX = b$) mede a sensibilidade da solução ($X$) a pequenas perturbações nos dados ($A$) ou ($b$). Sistemas <b style="color:#2ECC40">bem condicionados têm soluções estáveis</b>; sistemas <b style="color:#CC2E40">mal condicionados podem apresentar comportamento divergente</b> ou resultados imprecisos mesmo com erros pequenos.

### **1.2 Número de condição**

Para matrizes ($A$), definimos o número de condição como:
$\kappa(A) = ||A||.||A^{-1}||$.

Onde, qualquer norma matricial pode ser considerada, a mais usual é a norma-2 ou norma de Frobenius: $||A||_F  = ||A||_2 = \sqrt{\Sigma |a_{ij}|^2}$
- $\kappa(A) \approx 1$: sistema bem condicionado.
- $\kappa(A) \gg 1$: sistema mal condicionado.

In [9]:
import numpy as np

# Matriz quase singular
A = np.array([[1.0, 1.0], 
              [1.001, 1.0]])
b = np.array([10.0, 
              10.005])

cond_A = np.linalg.cond(A)
x = np.linalg.solve(A, b)

# Perturbação pequena em b
delta_b = np.array([0.0, 0.095])
b_perturbed = b + delta_b
x_perturbed = np.linalg.solve(A, b_perturbed)

print("A:")
print(A)
print(f"b = {b}")
print(f"b_pertubado = {b_perturbed}")
print("\nNúmero de condição cond(A):", cond_A)
print("\nSolução original (Ax=b) x =", x)
print("Solução com b pertubado x =", x_perturbed)
print("\nDiferença relativa em x:", np.linalg.norm(x_perturbed - x) / np.linalg.norm(x))
print("Diferença relativa em b:", np.linalg.norm(delta_b) / np.linalg.norm(b))

A:
[[1.    1.   ]
 [1.001 1.   ]]
b = [10.    10.005]
b_pertubado = [10.  10.1]

Número de condição cond(A): 4002.000750125229

Solução original (Ax=b) x = [5. 5.]
Solução com b pertubado x = [100. -90.]

Diferença relativa em x: 19.000000000002977
Diferença relativa em b: 0.006715835252641647


## 2. Decomposição em Valores Singulares

A decomposição em valores singulares (*Singular Value Decomposition* - SVD) fatoriza qualquer matriz real $A_{m \times n}$ na forma:

$$
A = U\Sigma V^T
$$
onde:
- $U$ é uma matriz ortogonal cujas colunas são os vetores singulares à esquerda;
- $V$ é uma matriz ortogonal cujas colunas são os vetores singulares à direita;
- $\Sigma$ é uma matriz diagonal, possivelmente retangular, cujos elementos diagonais $\sigma_i$ são os valores singulares de $A$, ou seja, a raiz dos autovalores de $A^TA$.

<center><img src="media/imgs/svd_0.png"></center>

### **2.1 Número de condição a partir de valores singulares**

Em termos de valores singulares o número de condição é:

$\kappa(A) = ||A||.||A^{-1}|| = max|\sigma_i|. max|\frac{1}{\sigma_i}| =  \sigma_{max}/\sigma_{min}$

Se algum valor singular $\sigma_i$ for nulo, $\sigma_{min}=0$ e então:

- $\kappa(A) = \infty$: sistema mal condicionado;
- $det(A)=0$;
- $A$ não possui inversa,


### **2.2 Abertura da decomposição em valores singulares**


Para uma matriz $A \in \mathbb{R}^{m \times n} \rightarrow U \in \mathbb{R}^{m \times m}, \Sigma \in \mathbb{R}^{m \times n}, V \in \mathbb{R}^{n \times n}$, a decomposição fica:

$$ \small
A =
\begin{bmatrix}
\vec{u}_1 & \vec{u}_2 & \cdots & \vec{u}_m
\end{bmatrix}
\begin{bmatrix}
\sigma_1 & 0 & \cdots & 0 & \cdots  & 0\\
0 & \sigma_2 & \cdots & 0 & \cdots & 0\\
\vdots & \vdots & \ddots & \vdots & \ddots & \vdots\\
0 & 0 & \cdots & \sigma_r & \cdots & 0\\
\vdots & \vdots & \ddots & \vdots& \ddots & \vdots\\
0 & 0 & \cdots & 0 & \cdots & \sigma_n \\
\hline \\
& & & [0]_{m-n\times n} & & 
\end{bmatrix}
\begin{bmatrix}
\vec{v}_1 & \vec{v}_2 & \cdots & \vec{v}_n
\end{bmatrix}^T
$$

onde:

- $r = posto(A)$;
- $\vec{u}_i$ e $\vec{v}_i$ são vetores singulares à esquerda e à direita, ou seja, autovetores de $AA^T$ e de  $A^TA$ respectivamente.

In [143]:
import sympy as sp
from IPython.display import display

import pandas as pd

# Matriz A com m = 4, n = 3 e posto r = 2
# A terceira coluna é a soma das duas primeiras
A = sp.Matrix([
    [1, 0, 1],
    [0, 1, 1],
    [1, 1, 2],
    [2, 1, 3]
])

# Calculando A^T A
ATA = A.T * A

# display(Math(r"A^T A ="))
# display(ATA)

# Autovalores de A^T A
autovalores = ATA.eigenvals()
# autovalores = sorted(autovalores, key=lambda x: float(x), reverse=True)


# Os valores singulares são as raízes quadradas dos autovalores de A^T A
singular_values = []

for lambd, multiplicidade in autovalores.items():
    for _ in range(multiplicidade):
        singular_values.append(sp.sqrt(lambd))

# Ordenando em ordem decrescente
singular_values = sorted(singular_values, key=lambda x: float(x), reverse=True)

df = pd.DataFrame([[A.rank(), str(sorted(autovalores, key=lambda x: float(x), reverse=True)), str(singular_values) ]], columns = ["rank(A)","Autovalores de A^T A", "Valores singulares" ])
# display(HTML(df.to_html(classes='table table-striped', index=False, border=2)))

# Construindo a matriz Sigma completa m x n
m, n = A.shape
Sigma = sp.zeros(m, n)

for i, sigma in enumerate(singular_values):
    Sigma[i, i] = sp.simplify(sigma)

# Para obter V, calculamos os autovetores de A^T A
eigen_data = ATA.eigenvects()

pares_autovalor_autovetor = []

for lambd, multiplicidade, autovetores in eigen_data:
    for vetor in autovetores:
        pares_autovalor_autovetor.append((lambd, vetor))

# Ordenando os pares pelo autovalor em ordem decrescente
pares_autovalor_autovetor = sorted(
    pares_autovalor_autovetor,
    key=lambda item: float(item[0]),
    reverse=True
)

# Normalizando os autovetores para formar os vetores singulares à direita
v_cols = []

for lambd, vetor in pares_autovalor_autovetor:
    v = vetor / sp.sqrt((vetor.T * vetor)[0])
    v_cols.append(sp.simplify(v))

# Matriz V: colunas são os vetores singulares à direita
V = sp.Matrix.hstack(*v_cols)

# ============================================================
# Cálculo dos vetores singulares à esquerda: colunas de U
# ============================================================

u_cols = []

for i, sigma in enumerate(singular_values):
    
    if sigma != 0:
        # Relação fundamental da SVD:
        # A v_i = sigma_i u_i
        u = A * V[:, i] / sigma
        u_cols.append(sp.simplify(u))

# Para os valores singulares nulos, completamos U com vetores do núcleo de A^T
# Esses vetores satisfazem A^T u = 0
null_AT = A.T.nullspace()

for vetor in null_AT:
    u = vetor / sp.sqrt((vetor.T * vetor)[0])
    u = sp.simplify(u)
    u_cols.append(u)

# Matriz U: colunas são os vetores singulares à esquerda
U = sp.Matrix.hstack(*u_cols)

display(Math(
    r"A = U \Sigma V^T"
))


display(Math(
    r"\small "+ sp.latex(A)
    + r" = "
    r"\underbrace{" + sp.latex(U) + r"}_{U}"
    r"\quad"
    r"\underbrace{" + sp.latex(Sigma) + r"}_{\Sigma}"
    r"\quad"
    r"\underbrace{" + sp.latex(V.T) + r"}_{V^T}"
))

# confere:
# display((U*Sigma*V.T).evalf())


<IPython.core.display.Math object>

<IPython.core.display.Math object>

### **2.3 Significado dos valores singulares nulos**

**Geometricamente**, valores singulares nulos significam que a matriz possui direções que são levadas para zero pela transformação linear.

Na SVD $A=U\Sigma V^T$, caso $r<n$, a matriz $\Sigma$ contém os valores singulares:

$$\sigma_1 \ge \sigma_2 \ge \dots \ge \sigma_r \ge 0 = \sigma_{r+1} = \sigma_{r+2} = \dots = \sigma_{n}$$
	​
Isso significa que os vetores singulares à direita $\vec{v}_{i}$ para $i>r$, ou seja, associados aos valores singulares nulos, representam o **espaço nulo de A**.

**Algebricamente**, valores singulares nulos significam que existe dependência linear, a matriz tem posto deficiente e, consequentemente, não possui inversa 

### **2.4 SVD compacta**

A SVD compacta usa apenas as colunas associadas aos valores singulares não nulos. Ou seja, até a coluna correspondente ao posto de A $(r)$:

$$
A =
U_r\Sigma_rV_r^T
$$

In [145]:
# Posto da matriz
r = A.rank()

# U_r: primeiras r colunas de U
U_r = U[:, :r]

# S_r: bloco diagonal r x r com valores singulares não nulos
S_r = Sigma[:r, :r]

# V_r: primeiras r colunas de V
V_r = V[:, :r]

# V_r^T: primeiras r linhas de V^T
V_r_T = V_r.T

display(Math(
    r"A = U_r \Sigma_r V_r^T"
))

display(Math(
    sp.latex(A)
    + r" = "
    r"\underbrace{" + sp.latex(U_r) + r"}_{U_r}"
    r"\quad"
    r"\underbrace{" + sp.latex(S_r) + r"}_{\Sigma_r}"
    r"\quad"
    r"\underbrace{" + sp.latex(V_r_T) + r"}_{V_r^T}"
))

# confere:
# display((U_r*S_r*V_r.T).evalf())

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### **2.4 Truncamento SVD**

No truncamento, em vez de usar todos os (r) valores singulares, mantemos apenas os primeiros (k), com $k < r$:

$$
A \approx A_k = U_k\Sigma_kV_k^T
$$

<center><img src="media/imgs/tsvd.png"></center>

In [146]:
import numpy as np

np.set_printoptions(precision=6, suppress=True)

# Matriz 4x4 simulada e quase singular: a 4a coluna e quase combinacao das outras.
A = np.array([[1.0, 2.0, 3.0, 6.0000],
              [2.0, 1.0, 0.0, 3.0001],
              [0.0, 1.0, 2.0, 3.0000],
              [1.0, 0.0, 1.0, 2.0001]])

# full_matrices=False retorna U, Sigma e Vt nas dimensões compactas.
# full_matrices=True  retorna U, Sigma e Vt nas dimensões cheias.
r = np.linalg.matrix_rank(A)
U, s, Vt = np.linalg.svd(A, full_matrices=True)
S = np.diag(s)
A_reconstruida = U @ S @ Vt

cond_svd = s.max() / s.min()
erro_reconstrucao = np.linalg.norm(A - A_reconstruida)

print("A:")
print(A)
print("\nValores singulares:")
print(s)
print(f"Numero de condicao (S.max() / S.min()): {cond_svd:.2f}")

k=r-2
print(f"\nValores singulares truncados (k={k}):")
s_trunc = s.copy()
s_trunc = s_trunc[:k]
print(s_trunc)
cond_k_svd = s_trunc.max() / s_trunc.min()
print(f"Numero de condicao (S_k.max() / S_k.min()): {cond_k_svd:.2f}" )

# U_k: primeiras k colunas de U
U_k = U[:, :k]
# S_k: bloco diagonal k x k com valores singulares não nulos
S_k = np.diag(s_trunc)
S_k = S_k[:k, :k]
# V_k: primeiras k colunas de V
V_k = Vt.T[:, :k]
# V_r^T: primeiras r linhas de V^T
V_k_T = V_k.T
# Aproximacao truncada
A_truncada = U_k @ S_k @ V_k_T

print("\nAproximacao truncada (A_k = U_k * S_k * V_k^T):")
print(A_truncada)
print("\nErro da aproximação ||A - A_k||:", np.linalg.norm(A - A_truncada))


A:
[[1.     2.     3.     6.    ]
 [2.     1.     0.     3.0001]
 [0.     1.     2.     3.    ]
 [1.     0.     1.     2.0001]]

Valores singulares:
[8.871322 2.166637 0.77867  0.000013]
Numero de condicao (S.max() / S.min()): 663874.89

Valores singulares truncados (k=2):
[8.871322 2.166637]
Numero de condicao (S_k.max() / S_k.min()): 4.09

Aproximacao truncada (A_k = U_k * S_k * V_k^T):
[[1.049699 1.885316 3.061116 5.996158]
 [2.068791 0.841284 0.084591 2.994776]
 [0.010211 0.976457 2.012556 2.999205]
 [0.7354   0.610515 0.674623 2.020572]]

Erro da aproximação ||A - A_k||: 0.7786698411879626


## 3. Matrizes pseudoinversas

A principal aplicação da decomposição em valores singulares é a redução de dimensionalidade com o descarte de autovalores e autovetores não importantes. 

Uma forma de aplicar esse efeito em solução de sistemas lineares é através da matriz pseudoinversa ($A^{+}$), cuja definição é:

$$
A=U_{r}\Sigma_{r}V_{r}^T \rightarrow A^{+} = V_r\Sigma_r^{-1}U_r^T
$$

onde $r=posto(A)$, indicando o ponto de truncamento das matrizes componentes da SVD.

Para um sistema linear cuja matriz de coeficientes $A$ possui posto completo, a pseudoinversa é $A^{+} = (A^TA)^{-1}A^T$, ou seja, coincide com a solução única do MMQ.

Para $A$ com deficiência de posto, $A^TA$ é singular, caso em que o MMQ tem infinitas soluções pois não basta impor a minimização dos resíduos, a pseudoinversa irá propiciar a unicidade de solução implicitamente impondo uma condição de contorno extra, que a energia do sistema ($||x||_2$) seja mínima:

$$
x_{\text{pseudo}} = \arg\min \Vert x \Vert_2 \quad \text{sujeito a} \quad x \in \arg\min \Vert Ax - b \Vert_2
$$



**Exercício 01 resolvido**: Resolver o sistema linear:

$$
\begin{bmatrix}
1 & 0 & 1 \\
0 & 1 & 1 
\end{bmatrix}\begin{bmatrix}
x_1 \\ x_2 \\ x_3 
\end{bmatrix}=\begin{bmatrix}
2 \\ 2
\end{bmatrix}
$$

🧐🧐🧐🧐🧐 Um sistema subdeterminado!?

**Solução exercício 01**

$det(A^TA) = \left|\begin{matrix}1 & 0 & 1 \\ 0 & 1 & 1 \\1 & 1 & 2 \end{matrix}\right| = 0 \rightarrow$ é inviável a solução via MMQ!

A solução pela Pseudoinversa é mais generalista e ainda robusta:

$Ax = b \rightarrow x = A^{+}b$

**Solução exercício 01**

In [ ]:
import numpy as np

A = np.array([[1.0, 0 , 1],
              [0, 1., 1]])

det_ATA = np.linalg.det(A.T @ A)
print("Determinante de A^T A:", det_ATA)

b = np.array([2.0, 2.0])

x = np.linalg.pinv(A) @ b  # Solução usando pseudoinversa
x

Determinante de A^T A: 0.0


array([0.66666667, 0.66666667, 1.33333333])

**Exercício 02**: Resolver o sistema linear:

$$
\begin{bmatrix}
1 & 1 \\
1 & 1.0001 \\
1 & 0.9999 
\end{bmatrix}\begin{bmatrix}
x_1 \\ x_2
\end{bmatrix}=\begin{bmatrix}
2 \\ 2.0001 \\ 1.9999
\end{bmatrix}
$$



**Exercício 03**: O princípio básico de posicionamento GNSS é a trilateração. Suponha que quatro satélites $S$ têm suas coordenadas sendo informadas pelo seu sinal GPS. Através da medição da pseudo-distância, o receptor estima sua distância a cada satélite. Como obter as coordenadas do receptor $P$?


<img src="media/imgs/img12.png" width=600>

$\begin{cases}
d_1 = ||P-S_1|| \ \ \rightarrow (f_1) \\
d_2 = ||P-S_2|| \ \ \rightarrow (f_2) \\
d_3 = ||P-S_3|| \ \ \rightarrow (f_3) \\ 
d_4 = ||P-S_4|| \ \ \rightarrow (f_4) \\ 
\end{cases}$

Faça a solução primeiramente para os dados tachados, onde o sistema não apresentará problemas de condicionamento, em seguida use os dados propostos e tente resolver utilizando a pseudoinversa em seus cálculos:

<pre>
dados = {"X_1": 0000.0, "Y_1": 5000.0, "Z_1": 4000.0, 
         <span style="text-decoration: line-through;">"X_2": 6000.0, "Y_2": 4000.0, "Z_2": 5000.0, </span>
         "X_2": 0000.0, "Y_2": 5000.0, "Z_2": 4000.0,
         "X_3": 7000.0, "Y_3": -3000.0, "Z_3": 3500.0,
         <span style="text-decoration: line-through;">"X_4": -2000.0, "Y_4": -2000.0, "Z_4": 3000.0,</span>
         "X_4": 7000.0, "Y_4": -3000.0, "Z_4": 3500.0,
         "d_1": 5724.836, 
         <span style="text-decoration: line-through;">"d_2": 6229.272,  </span>
         "d_2": 5725.0, 
         "d_3": 6630.869, 
         <span style="text-decoration: line-through;">"d_4": 5914.816, </span>
         "d_4": 6631, 
        }
</pre>

## Complemento: Resumo das soluções de sistemas lineares

Para um sistema linear $A_{m \times n}x=b$,  com $posto(A)=r$:

1. Matriz quadrada e posto cheio $m=n=r$
    - Solução trivial : $x=A^{-1}b$
    - Restrição: Não há
2. Sistema sobredeterminado e posto cheio $m>n=r$
    - Solução de mínimos quadrados: $x=(A^TA)^{-1}A^Tb$
    - Restrição nos resíduos: $x=\arg\min \Vert Ax-b\Vert_2$
3. Matriz posto deficiente ou caso subdeterminado $m>n>r$
    - Solução pela pseudoinversa: $x=A^+b$
    - Restrição nos resíduos e nos argumentos: $x_{\text{pseudo}} = \arg\min \Vert x \Vert_2 \quad \text{sujeito a} \quad x \in \arg\min \Vert Ax - b \Vert_2$

<!-- **Exercício 04 gabarito** 

```python
import sympy as sp
from sympy import symbols, Matrix, sqrt, simplify, diff
from IPython.display import display
import matplotlib.pyplot as plt

dados = {"X_1": 0000.0, "Y_1": 5000.0, "Z_1": 4000.0, 
        #  "X_2": 6000.0, "Y_2": 4000.0, "Z_2": 5000.0,
         "X_2": 0000.0, "Y_2": 5000.0, "Z_2": 4000.0,
         "X_3": 7000.0, "Y_3": -3000.0, "Z_3": 3500.0,
        #  "X_4": -2000.0, "Y_4": -2000.0, "Z_4": 3000.0,
         "X_4": 7000.0, "Y_4": -3000.0, "Z_4": 3500.0,
         "d_1": 5724.836, 
        #  "d_2": 6229.272, 
         "d_2": 5725.0, 
         "d_3": 6630.869, 
        #  "d_4": 5914.816, 
         "d_4": 6631.0, 
         }

def norma(A,B):
    return sqrt((A-B).dot(A-B))

# Definindo as variáveis simbólicas
S1 = Matrix([dados["X_1"], dados["Y_1"], dados["Z_1"]])
S2 = Matrix([dados["X_2"], dados["Y_2"], dados["Z_2"]])
S3 = Matrix([dados["X_3"], dados["Y_3"], dados["Z_3"]])
S4 = Matrix([dados["X_4"], dados["Y_4"], dados["Z_4"]])
X_p, Y_p, Z_p = symbols('X_p Y_p Z_p')
P = Matrix([X_p, Y_p, Z_p])

# funções
f1 = norma(P,S1)
f2 = norma(P,S2)
f3 = norma(P,S3)
f4 = norma(P,S4)
F = Matrix([[f1],
            [f2],
            [f3],
            [f4]])
# display(F)

# Derivadas parciais
df1_Xp, df1_Yp, df1_Zp = diff(f1, X_p), diff(f1, Y_p), diff(f1, Z_p)
df2_Xp, df2_Yp, df2_Zp = diff(f2, X_p), diff(f2, Y_p), diff(f2, Z_p)
df3_Xp, df3_Yp, df3_Zp = diff(f3, X_p), diff(f3, Y_p), diff(f3, Z_p)
df4_Xp, df4_Yp, df4_Zp = diff(f4, X_p), diff(f4, Y_p), diff(f4, Z_p)

# Montando a matriz Jacobiana
A = Matrix([[df1_Xp, df1_Yp, df1_Zp],
            [df2_Xp, df2_Yp, df2_Zp],
            [df3_Xp, df3_Yp, df3_Zp],
            [df4_Xp, df4_Yp, df4_Zp]]) 

# Inicialização de X0
X_0 = {X_p: 0.0, Y_p: 0.0, Z_p: 0.0} 

### Início do ajustamento 

# mount Lb
Lb = np.array([[dados["d_1"]],
               [dados["d_2"]],
               [dados["d_3"]],
               [dados["d_4"]]])

values = {**dados, **X_0}

erros = [] # guardar cada erro durante as iterações para compor um gráfico
erro = 10000000
k = 0
while erro>10e-6:
    # update L0 = F(X0)
    L0 = np.array(F.subs(values).evalf().tolist(), dtype=float)
    # update L
    L = L0 - Lb
    # update A
    a = np.array(A.subs(values).evalf().tolist(), dtype=float)
    # update X = vetor das correções
    # DeltaX = -np.linalg.inv(a.T @ a) @ a.T @ L
    DeltaX = -np.linalg.pinv(a) @ L
    DeltaX = DeltaX.reshape(-1) # reshape para converter de matriz coluna para vetor
    # update erro
    erro = np.linalg.norm(DeltaX)
    erros.append(erro)
    # update X0
    X0 = DeltaX + np.array(list(X_0.values()))
    # print("\nXa:\n", X0)
    X_0 = {X_p: X0[0], Y_p: X0[1], Z_p: X0[2]} 
    print(f"Iter {k}: \tErro: {erro} \t- X0: {X0}")
    # update lista de valores para a próxima iteração
    values = {**dados, **X_0}
    k+=1

# Gráfico
plt.title("Convergência do erro no ajustamento")
plt.xlabel("Iterações")
plt.ylabel("Erro")
plt.grid()
plt.plot(erros)
plt.show()``` -->